In [2]:
from datasets import load_dataset
from itertools import islice
dataset = load_dataset(
    "eltorio/ROCOv2-radiology",
    streaming = True,
    split = "train"
)

dataset_shuffle = dataset.shuffle(seed = 42, buffer_size = 5000)
samples = list(islice(dataset_shuffle,1000))

g:\NeuroMed-VLM\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
image = samples[0]["image"].convert("RGB")

pixel_values = transform(image)

print(pixel_values.shape)
pixel_values = pixel_values.unsqueeze(dim=0)

print(pixel_values.shape)

torch.Size([3, 224, 224])
torch.Size([1, 3, 224, 224])


In [5]:
from torch.utils.data import Dataset

class MedicalDataset(Dataset):
    def __init__(self,samples,transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return(len(self.samples))

    def __getitem__(self,index):
        sample = self.samples[index]
        return {
            "image": self.transform(sample["image"].convert("RGB")),
            "caption": sample["caption"],
            "image_id": sample["image_id"]
        }

In [6]:
dataset = MedicalDataset(samples, transform=transform)

In [7]:
def collate_fn(batch,tokenizer):
    images = [sample["image"] for sample in batch]
    captions = [sample["caption"] for sample in batch]
    image_ids = [sample["image_id"] for sample in batch]
    image_batch = torch.stack(images,dim = 0)
    encoded = tokenizer(
        captions,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    return{ "pixel_values": image_batch,
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "image_ids": image_ids}

In [8]:
from torch.utils.data import DataLoader
from functools import partial
from transformers import AutoTokenizer
text_model_name = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)
tokenizer = AutoTokenizer.from_pretrained(text_model_name)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=partial(collate_fn, tokenizer=tokenizer)
)


In [9]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "facebook/dinov2-small"
)

print("Model type:", config.model_type)
print("Hidden size:", config.hidden_size)
print("Patch size:", config.patch_size)
print("Layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("Image size:", config.image_size)

Model type: dinov2
Hidden size: 384
Patch size: 14
Layers: 12
Attention heads: 6
Image size: 518


In [10]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained("facebook/dinov2-small")

print(type(model))
print("Total parameters:", sum(p.numel() for p in model.parameters()))

model.eval()

Loading weights: 100%|██████████| 223/223 [00:00<00:00, 4565.43it/s]

<class 'transformers.models.dinov2.modeling_dinov2.Dinov2Model'>
Total parameters: 22056576


Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (k_proj): Linear(in_features=384, out_features=384, bias=True)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): Linear(in_features=384, out_features=384, bias=True)
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): Dinov2MLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=

In [11]:
from transformers import AutoConfig

text_model_name = (
    "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
)

text_config = AutoConfig.from_pretrained(text_model_name)

print("Model type:", text_config.model_type)
print("Hidden size:", text_config.hidden_size)
print("Layers:", text_config.num_hidden_layers)
print("Attention heads:", text_config.num_attention_heads)
print("Vocabulary size:", text_config.vocab_size)
print("Max positions:", text_config.max_position_embeddings)

Model type: bert
Hidden size: 768
Layers: 12
Attention heads: 12
Vocabulary size: 30522
Max positions: 512


In [12]:
from transformers import AutoModel

text_model = AutoModel.from_pretrained(text_model_name)
text_model.eval()

print("Total parameters:", sum(p.numel() for p in text_model.parameters()))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18094.97it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total parameters: 109482240


In [13]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

image_projection = nn.Linear(384, 256)
text_projection = nn.Linear(768, 256)

optimizer = optim.AdamW(
    list(image_projection.parameters())
    + list(text_projection.parameters()),
    lr=1e-4
)

temperature = 0.07

for step, batch in enumerate(loader):


    images = batch["pixel_values"]
    input_ids = batch["input_ids"]

    assert images.shape[1:] == (3, 224, 224)
    assert images.shape[0] == input_ids.shape[0]
    assert input_ids.shape[1] <= 128

    with torch.no_grad():
        outputs = model(pixel_values=batch["pixel_values"])
    hidden = outputs.last_hidden_state

    cls_embedding = hidden[:, 0, :]

    with torch.no_grad():
        text_outputs = text_model(input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"]
                        )
    hidden = text_outputs.last_hidden_state

    text_cls = hidden[:, 0, :]
    mask = batch["attention_mask"]
    mask_expanded = mask.unsqueeze(-1)
    masked_hidden = hidden * mask_expanded
    sum_hidden = masked_hidden.sum(dim=1)
    token_counts = mask_expanded.sum(dim=1)
    text_mean = sum_hidden / token_counts.clamp(min=1)

    image_features = cls_embedding
    text_features = text_mean

    image_projected = image_projection(image_features)
    text_projected = text_projection(text_features)

    image_norm = F.normalize(image_projected, p=2, dim=-1)
    text_norm = F.normalize(text_projected, p=2, dim=-1)

    similarity = image_norm @ text_norm.T
    logits = similarity / temperature

    labels = torch.arange(images.shape[0])
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    loss = (loss_i2t + loss_t2i) / 2

    optimizer.zero_grad()
    loss.backward()
    
    print(
    f"Step {step+1}: "
    f"Loss={loss.item():.4f}, "
    f"Image grad={image_projection.weight.grad.norm().item():.4f}, "
    f"Text grad={text_projection.weight.grad.norm().item():.4f}"
)
    optimizer.step()

    if step == 4:
        break


Step 1: Loss=1.6514, Image grad=5.4343, Text grad=7.8369
Step 2: Loss=1.3905, Image grad=1.7802, Text grad=2.3495
Step 3: Loss=1.3884, Image grad=1.9234, Text grad=2.8035
Step 4: Loss=1.4680, Image grad=3.0151, Text grad=4.4515
Step 5: Loss=1.5780, Image grad=2.9082, Text grad=4.3322


In [15]:
fixed_batch = next(iter(loader))

for step in range(50):
    batch = fixed_batch
    images = batch["pixel_values"]
    input_ids = batch["input_ids"]

    assert images.shape[1:] == (3, 224, 224)
    assert images.shape[0] == input_ids.shape[0]
    assert input_ids.shape[1] <= 128

    with torch.no_grad():
        outputs = model(pixel_values=batch["pixel_values"])
    hidden = outputs.last_hidden_state

    cls_embedding = hidden[:, 0, :]

    with torch.no_grad():
        text_outputs = text_model(input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"]
                        )
    hidden = text_outputs.last_hidden_state

    text_cls = hidden[:, 0, :]
    mask = batch["attention_mask"]
    mask_expanded = mask.unsqueeze(-1)
    masked_hidden = hidden * mask_expanded
    sum_hidden = masked_hidden.sum(dim=1)
    token_counts = mask_expanded.sum(dim=1)
    text_mean = sum_hidden / token_counts.clamp(min=1)

    image_features = cls_embedding
    text_features = text_mean

    image_projected = image_projection(image_features)
    text_projected = text_projection(text_features)

    image_norm = F.normalize(image_projected, p=2, dim=-1)
    text_norm = F.normalize(text_projected, p=2, dim=-1)

    similarity = image_norm @ text_norm.T
    logits = similarity / temperature

    labels = torch.arange(images.shape[0])
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    loss = (loss_i2t + loss_t2i) / 2

    optimizer.zero_grad()
    loss.backward()

    print(
    f"Step {step+1}: "
    f"Loss={loss.item():.4f}, "
    f"Image grad={image_projection.weight.grad.norm().item():.4f}, "
    f"Text grad={text_projection.weight.grad.norm().item():.4f}"
)
    optimizer.step()

Step 1: Loss=1.4551, Image grad=2.2656, Text grad=3.2837
Step 2: Loss=1.4330, Image grad=2.1448, Text grad=3.1147
Step 3: Loss=1.3967, Image grad=1.9875, Text grad=2.8955
Step 4: Loss=1.3514, Image grad=1.8284, Text grad=2.6737
Step 5: Loss=1.3009, Image grad=1.6790, Text grad=2.4629
Step 6: Loss=1.2478, Image grad=1.5415, Text grad=2.2653
Step 7: Loss=1.1942, Image grad=1.4265, Text grad=2.1017
Step 8: Loss=1.1412, Image grad=1.3550, Text grad=2.0126
Step 9: Loss=1.0895, Image grad=1.3395, Text grad=2.0223
Step 10: Loss=1.0390, Image grad=1.3640, Text grad=2.1023
Step 11: Loss=0.9894, Image grad=1.3922, Text grad=2.1894
Step 12: Loss=0.9403, Image grad=1.3931, Text grad=2.2306
Step 13: Loss=0.8916, Image grad=1.3536, Text grad=2.2040
Step 14: Loss=0.8436, Image grad=1.2782, Text grad=2.1167
Step 15: Loss=0.7969, Image grad=1.1829, Text grad=1.9945
Step 16: Loss=0.7520, Image grad=1.0879, Text grad=1.8697
Step 17: Loss=0.7093, Image grad=1.0116, Text grad=1.7711
Step 18: Loss=0.6689, I

In [10]:
hidden = outputs.last_hidden_state

cls_embedding = hidden[:, 0, :]

patch_embeddings = hidden[:, 1:, :]

print("CLS:", cls_embedding.shape)
print("Patches:", patch_embeddings.shape)

CLS: torch.Size([1, 384])
Patches: torch.Size([1, 256, 384])


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(text_model_name)

caption = "Axial brain MRI showing an intracranial lesion."

encoded = tokenizer(
    caption,
    return_tensors="pt"
)

with torch.no_grad():
    text_outputs = text_model(**encoded)

print("Input IDs:", encoded["input_ids"].shape)
print("Hidden states:", text_outputs.last_hidden_state.shape)

Input IDs: torch.Size([1, 10])
Hidden states: torch.Size([1, 10, 768])


In [15]:
hidden = text_outputs.last_hidden_state

text_cls = hidden[:, 0, :]
mask = encoded["attention_mask"]
mask_expanded = mask.unsqueeze(-1)

print("CLS shape:", text_cls.shape)
print("Expanded mask:", mask_expanded.shape)

CLS shape: torch.Size([1, 768])
Expanded mask: torch.Size([1, 10, 1])


In [16]:
# Step 1: Mask掉Padding位置
masked_hidden = hidden * mask_expanded

# Step 2: 沿Token维度求和
sum_hidden = masked_hidden.sum(dim=1)

# Step 3: 计算每条Caption的有效Token数量
token_counts = mask_expanded.sum(dim=1)

# Step 4: 求平均
text_mean = sum_hidden / token_counts

print("Sum shape:", sum_hidden.shape)
print("Counts shape:", token_counts.shape)
print("Mean shape:", text_mean.shape)

Sum shape: torch.Size([1, 768])
Counts shape: torch.Size([1, 1])
Mean shape: torch.Size([1, 768])


In [17]:
import torch.nn as nn

image_projection = nn.Linear(384, 256)
text_projection = nn.Linear(768, 256)

image_features = cls_embedding
text_features = text_mean

image_projected = image_projection(image_features)
text_projected = text_projection(text_features)

print("Image:", image_projected.shape)
print("Text:", text_projected.shape)

Image: torch.Size([1, 256])
Text: torch.Size([1, 256])


In [18]:
import torch.nn.functional as F
image_norm = F.normalize(image_projected, p=2, dim=-1)
text_norm = F.normalize(text_projected, p=2, dim=-1)

print("Image norm:", image_norm.norm(dim=1))
print("Text norm:", text_norm.norm(dim=1))
similarity = image_norm @ text_norm.T

print("Similarity shape:", similarity.shape)
print("Similarity:", similarity.item())

Image norm: tensor([1.], grad_fn=<LinalgVectorNormBackward0>)
Text norm: tensor([1.], grad_fn=<LinalgVectorNormBackward0>)
Similarity shape: torch.Size([1, 1])
Similarity: -0.0568389855325222
